# Khmer Math Lab — Intent Classifier Training

This notebook trains a Transformer-based sequence classification model to detect user mathematical intent in **Khmer and English**:
- `solve_equation` (ដោះស្រាយសមីការ)
- `evaluate_expression` (គណនាតម្លៃកន្សោម)
- `simplify_expression` (សម្រួលកន្សោម)
- `unknown` (សំណួរទូទៅ / មិនច្បាស់លាស់)

In [1]:
# 1. Environment check
import torch
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

Using device: mps


## 2. Prepare Training & Validation Data
We define bilingual training examples for each intent class.

In [2]:
import json
import random
from datasets import Dataset

INTENT_LABELS = [
    "solve_equation",
    "evaluate_expression",
    "simplify_expression",
    "unknown"
]
label2id = {name: idx for idx, name in enumerate(INTENT_LABELS)}
id2label = {idx: name for idx, name in enumerate(INTENT_LABELS)}

# Sample bilingual dataset
raw_samples = [
    # SOLVE_EQUATION
    ("ដោះស្រាយ 2x + 5 = 15", "solve_equation"),
    ("រកតម្លៃ x នៃសមីការ 3x - 9 = 0", "solve_equation"),
    ("ជួយដោះស្រាយ x^2 - 4 = 0", "solve_equation"),
    ("solve for x: 5x + 10 = 25", "solve_equation"),
    ("find the root of 2x = 8", "solve_equation"),
    ("solve 4x - 7 = 3x + 2", "solve_equation"),
    ("កំណត់តម្លៃ y ពី 2y + 1 = 7", "solve_equation"),
    ("solve equation x + 3 = 10", "solve_equation"),
    # EVALUATE_EXPRESSION
    ("គណនា 25 + 40 * 2", "evaluate_expression"),
    ("គិតតម្លៃ 3/4 + 1/2", "evaluate_expression"),
    ("តើ 15 ភាគរយ នៃ 200 ស្មើប៉ុន្មាន", "evaluate_expression"),
    ("evaluate 12 * (5 + 3)", "evaluate_expression"),
    ("calculate 100 / 4 + 7", "evaluate_expression"),
    ("compute 2^3 + 5", "evaluate_expression"),
    ("what is 50% of 80", "evaluate_expression"),
    ("គណនាតម្លៃ 10 - 3 * 2", "evaluate_expression"),
    # SIMPLIFY_EXPRESSION
    ("សម្រួលកន្សោម 2(x + 3) + 4x", "simplify_expression"),
    ("បង្រួមកន្សោម 3x + 5x - 2x", "simplify_expression"),
    ("simplify 4x + 7x - 2", "simplify_expression"),
    ("reduce the fraction 18/24", "simplify_expression"),
    ("simplify (x^2 - 4)/(x - 2)", "simplify_expression"),
    ("សម្រួល (x + 1)(x - 1)", "simplify_expression"),
    # UNKNOWN
    ("សួស្តី តើអ្នកសុខសប្បាយជាទេ?", "unknown"),
    ("តើអ្នកជានរណា?", "unknown"),
    ("hello how are you", "unknown"),
    ("who created this math app", "unknown"),
]

# Expand with synthetic variations
samples = []
for text, intent in raw_samples:
    samples.append({"text": text, "label": label2id[intent]})

random.seed(42)
random.shuffle(samples)
split_idx = int(len(samples) * 0.8)

train_data = Dataset.from_list(samples[:split_idx])
val_data = Dataset.from_list(samples[split_idx:])
print(f"Train examples: {len(train_data)}, Validation examples: {len(val_data)}")

/Users/kiddd/Development/math-lab/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train examples: 20, Validation examples: 6


## 3. Load Pretrained Tokenizer & Model
We use `xlm-roberta-base` or a lightweight multilingual BERT model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Use lightweight multilingual model suitable for Khmer and English
MODEL_NAME = "bert-base-multilingual-cased"

print(f"Loading tokenizer and model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(INTENT_LABELS),
    id2label=id2label,
    label2id=label2id,
)

Loading tokenizer and model: bert-base-multilingual-cased...


## 4. Tokenization

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_data.map(tokenize_fn, batched=True)
tokenized_val = val_data.map(tokenize_fn, batched=True)

## 5. Training Setup

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = np.mean(preds == labels)
    return {"accuracy": float(acc)}

training_args = TrainingArguments(
    output_dir="./models/intent_classifier_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Ready to train.")
# trainer.train()

## 6. Save Model and Test Inference

In [ ]:
# Save final checkpoint
SAVE_DIR = "./models/intent_classifier_final"
# trainer.save_model(SAVE_DIR)
# tokenizer.save_pretrained(SAVE_DIR)

# Quick zero-shot / rule-based fallback test
test_queries = [
    "ដោះស្រាយ 3x + 12 = 0",
    "គណនា 45 * 2 + 10",
    "សម្រួលកន្សោម 5x + 3x",
    "How to solve quadratic equations?"
]

print("Sample queries ready for testing:")
for q in test_queries:
    print(" -", q)